# Fine-Tuning for Chatbot Q&A

Fine-tunes the pre-trained transformer on personal details Q&A pairs.

**What this does:**
- Loads the pre-trained checkpoint (from TinyStories training)
- Fine-tunes on `Question: {q} Answer: {a}<EOS>` format
- Uses **masked loss** — only answer tokens contribute to the gradient
- Result: a chatbot that answers questions about Abhay Chaturvedi

**Prerequisites:** Upload the full repo to Colab (or mount via Google Drive).  
Required files: `src/` directory, `dataset/tokenizer.json`, `dataset/personal-details/`, pre-trained checkpoint.

In [ ]:
# PyTorch is pre-installed on Colab — just verify GPU
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
PATH_TO_REPO = "/content/vSLM-From-Scratch/"

import sys
sys.path.insert(0, PATH_TO_REPO)

In [ ]:
from src.model.config import ModelConfig
from src.model.transformer import Transformer
from src.data.finetune_dataset import FineTuneDataset
from src.training.finetuner import FineTuner, FineTuneConfig
from src.tokenization.tokenizer import BPETokenizer

from pathlib import Path

## Google Drive Setup

Mount Google Drive for checkpoint persistence and data access.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

GDRIVE_DATA_DIR = Path("/content/drive/MyDrive/vSLM-data")
GDRIVE_CKPT_DIR = Path("/content/drive/MyDrive/vSLM-checkpoints")
GDRIVE_FT_CKPT_DIR = Path("/content/drive/MyDrive/vSLM-checkpoints-finetune")
GDRIVE_FT_CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Data dir:             {GDRIVE_DATA_DIR}")
print(f"Pre-train checkpoints: {GDRIVE_CKPT_DIR}")
print(f"Fine-tune checkpoints: {GDRIVE_FT_CKPT_DIR}")

In [ ]:
# Find the best pre-trained checkpoint to start from
def find_latest_checkpoint(ckpt_dir, prefix="step_"):
    ckpts = list(ckpt_dir.glob(f"{prefix}*.pt"))
    if ckpts:
        ckpts.sort(key=lambda p: int(p.stem.split("_")[-1]))
        return ckpts[-1]
    final = ckpt_dir / "final.pt"
    return final if final.exists() else None

pretrained_ckpt = find_latest_checkpoint(GDRIVE_CKPT_DIR)
if pretrained_ckpt:
    print(f"Pre-trained checkpoint: {pretrained_ckpt}")
else:
    print("WARNING: No pre-trained checkpoint found! Fine-tuning from random init.")

# Also check if there's an existing fine-tune checkpoint to resume from
ft_resume_ckpt = find_latest_checkpoint(GDRIVE_FT_CKPT_DIR, prefix="finetune_step_")
if ft_resume_ckpt:
    print(f"Fine-tune checkpoint to resume: {ft_resume_ckpt}")

## 1. Load Tokenizer

In [ ]:
tokenizer = BPETokenizer.load(str(GDRIVE_DATA_DIR / "tokenizer.json"))
print(f"Vocab size: {len(tokenizer.vocab)}")

## 2. Load Personal Details Q&A Dataset

Loads all CSV files from `dataset/personal-details/` and formats them as chatbot Q&A pairs:
```
Question: {question} Answer: {answer}<EOS>
```

Uses **loss masking** so only the answer portion contributes to the gradient.

In [ ]:
CONTEXT_LENGTH = 128

# Personal details CSVs — either from repo or from GDrive
personal_csv_dir = Path(PATH_TO_REPO) / "dataset" / "personal-details"
if not personal_csv_dir.exists():
    personal_csv_dir = GDRIVE_DATA_DIR / "personal-details"

dataset = FineTuneDataset.from_csv_dir(
    personal_csv_dir, tokenizer, context_length=CONTEXT_LENGTH,
)
train_data, val_data = dataset.split(val_fraction=0.1)

In [ ]:
# Inspect a sample batch
x, y, mask = train_data.get_batch(batch_size=4, device="cpu")
print(f"Input shape:     {x.shape}")
print(f"Target shape:    {y.shape}")
print(f"Loss mask shape: {mask.shape}")
print(f"\nMask[0] nonzero: {mask[0].sum().item():.0f} answer tokens out of {CONTEXT_LENGTH}")

# Decode a sample to see the Q&A format
ids = x[0].tolist()
# Remove padding
ids = [t for t in ids if t != 0 or ids.index(0) > ids.index(t)]
print(f"\nDecoded example: {tokenizer.decode(ids)}")

## 3. Configure Model

Same architecture as pre-training — must match the checkpoint.

In [ ]:
model_config = ModelConfig(
    vocab_size=len(tokenizer.vocab),  # 4096
    context_length=CONTEXT_LENGTH,    # 128
    d_model=256,
    n_heads=4,
    n_layers=4,
    d_ff=1024,
    dropout_rate=0.1,
)

model = Transformer(model_config)
print(model_config)
print(f"Parameters: {model.count_parameters():,}")

## 4. Configure Fine-Tuning

Key differences from pre-training:
- **Much lower learning rate** (5e-5 vs 6e-4) — prevents catastrophic forgetting
- **Fewer steps** (3000 vs 50,000) — small dataset converges fast
- **Smaller batch size** (32 vs 256) — fewer examples available
- **Masked loss** — only answer tokens contribute to the gradient

In [ ]:
ft_config = FineTuneConfig(
    batch_size=32,
    learning_rate=5e-5,              # low LR to preserve pre-trained knowledge
    warmup_steps=100,
    max_steps=3000,
    weight_decay=0.01,
    grad_clip=1.0,
    log_every=50,
    eval_every=500,
    eval_batches=10,
    checkpoint_every=1000,
    checkpoint_dir=str(GDRIVE_FT_CKPT_DIR),
    seed=42,
    use_amp=True,
    compile_model=True,
    sample_prompts=(
        "Question: Who is Abhay? Answer:",
        "Question: What is Abhay's current role? Answer:",
        "Question: hi Answer:",
    ),
    sample_max_tokens=80,
    sample_temperature=0.7,
    sample_top_k=40,
)

finetuner = FineTuner(model, model_config, ft_config)

## 5. Fine-Tune

In [ ]:
history = finetuner.finetune(
    train_data,
    val_data,
    pretrained_checkpoint=pretrained_ckpt,
    tokenizer=tokenizer,
)

## 6. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history["step"], history["train_loss"])
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].set_title("Fine-Tuning Training Loss")

if history["val_loss"]:
    eval_steps = [s for s in history["step"] if s % ft_config.eval_every == 0]
    axes[1].plot(eval_steps[: len(history["val_loss"])], history["val_loss"])
    axes[1].set_xlabel("Step")
    axes[1].set_ylabel("Loss")
    axes[1].set_title("Fine-Tuning Validation Loss")

plt.tight_layout()
plt.show()

## 7. Test the Chatbot

Try various question types to verify fine-tuning worked.

In [ ]:
test_prompts = [
    # Personal info
    "Question: Who is Abhay Chaturvedi? Answer:",
    "Question: What is Abhay's email? Answer:",
    "Question: Where does Abhay work? Answer:",
    # Skills & experience
    "Question: What are Abhay's skills? Answer:",
    "Question: What did Abhay build at Pegasystems? Answer:",
    "Question: What is Abhay's education? Answer:",
    # Greetings & meta
    "Question: hi Answer:",
    "Question: who are you Answer:",
    # Boundary
    "Question: Can you write code for me? Answer:",
    # Story (should still work from pre-training)
    "Once upon a time",
]

for prompt in test_prompts:
    text = finetuner.generate(
        tokenizer, prompt,
        max_tokens=150, temperature=0.7, top_k=40,
    )
    print(f"--- {prompt} ---")
    print(text)
    print()

In [ ]:
# Interactive: try your own prompt
text = finetuner.generate(
    tokenizer,
    prompt="Question: Tell me about Abhay's projects. Answer:",
    max_tokens=200,
    temperature=0.7,
    top_k=50,
)
print(text)